# 01. Загрузка и проверка данных

Ноутбук загружает один Excel/CSV-файл транзакций из `data/raw/`, нормализует названия колонок, выполняет проверки качества и собирает дневную витрину `sales_date × stock_code × market_id`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_FILE_CANDIDATES, INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from src.data_checks import (
    find_raw_data_file,
    normalize_column_names,
    run_all_data_checks,
)

## Загрузка файла

Поддерживаются `online_retail_II.xlsx`, `online_retail.xlsx`, `online_retail_II.csv`, `online_retail.csv`.

In [ ]:
raw_path = find_raw_data_file(RAW_FILE_CANDIDATES)

if raw_path.suffix.lower() in ['.xlsx', '.xls']:
    sheets = pd.read_excel(raw_path, sheet_name=None)
    raw_transactions = pd.concat(sheets.values(), ignore_index=True)
elif raw_path.suffix.lower() == '.csv':
    raw_transactions = pd.read_csv(raw_path)
else:
    raise ValueError('Поддерживаются только Excel и CSV.')

transactions = normalize_column_names(raw_transactions)
transactions.head()

## Подготовка полей

Возвратные и отмененные операции не удаляются. Они помечаются отдельным флагом и учитываются в `net_sales_qty`.

In [ ]:
transactions['invoice'] = transactions['invoice'].astype(str)
transactions['stock_code'] = transactions['stock_code'].astype('string').str.strip()
transactions['description'] = transactions['description'].astype('string').str.strip()
transactions['country'] = transactions['country'].astype('string').str.strip()
transactions['quantity'] = pd.to_numeric(transactions['quantity'], errors='coerce')
transactions['unit_price'] = pd.to_numeric(transactions['unit_price'], errors='coerce')
transactions['invoice_date'] = pd.to_datetime(transactions['invoice_date'], errors='coerce')

transactions['sales_date'] = transactions['invoice_date'].dt.date
transactions['market_id'] = transactions['country']
transactions['is_return_flag'] = transactions['invoice'].str.startswith('C', na=False) | (transactions['quantity'] < 0)
transactions['is_price_missing'] = transactions['unit_price'].isna()
transactions['is_price_invalid'] = transactions['unit_price'].isna() | (transactions['unit_price'] <= 0)
transactions['revenue'] = transactions['quantity'] * transactions['unit_price']

transactions.head()

## Проверки качества

In [ ]:
quality_checks = run_all_data_checks(transactions)
quality_checks

## Сохранение подготовленной таблицы

In [ ]:
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
prepared_columns = [
    'invoice',
    'stock_code',
    'description',
    'quantity',
    'invoice_date',
    'sales_date',
    'unit_price',
    'customer_id',
    'country',
    'market_id',
    'is_return_flag',
    'is_price_missing',
    'is_price_invalid',
    'revenue',
]
transactions_clean = transactions[prepared_columns].copy()
transactions_clean.to_csv(INTERIM_DATA_DIR / 'transactions_clean.csv', index=False)
quality_checks.to_csv(INTERIM_DATA_DIR / 'data_quality_checks.csv', index=False)
transactions_clean.head()

## Дневная витрина продаж

Целевой уровень: одна строка = `sales_date × stock_code × market_id`.

In [ ]:
mart_source = transactions_clean.dropna(subset=['sales_date', 'stock_code', 'market_id']).copy()

mart_daily_sales = (
    mart_source
    .groupby(['sales_date', 'stock_code', 'market_id'], as_index=False)
    .agg(
        description=('description', 'last'),
        sales_qty=('quantity', lambda values: values[~mart_source.loc[values.index, 'is_return_flag']].sum()),
        avg_unit_price=('unit_price', lambda values: values[values > 0].mean()),
        revenue=('revenue', 'sum'),
        invoices_cnt=('invoice', 'nunique'),
        customers_cnt=('customer_id', 'nunique'),
        returns_qty=('quantity', lambda values: abs(values[mart_source.loc[values.index, 'is_return_flag']].sum())),
        net_sales_qty=('quantity', 'sum'),
        is_return_flag=('is_return_flag', 'max'),
        is_price_missing=('is_price_missing', 'max'),
    )
)

mart_daily_sales['sales_date'] = pd.to_datetime(mart_daily_sales['sales_date'])
mart_daily_sales['weekday'] = mart_daily_sales['sales_date'].dt.dayofweek
mart_daily_sales['month'] = mart_daily_sales['sales_date'].dt.month
mart_daily_sales['year'] = mart_daily_sales['sales_date'].dt.year
mart_daily_sales['is_weekend'] = mart_daily_sales['weekday'].isin([5, 6])

mart_daily_sales = mart_daily_sales[
    [
        'sales_date',
        'stock_code',
        'description',
        'market_id',
        'sales_qty',
        'avg_unit_price',
        'revenue',
        'invoices_cnt',
        'customers_cnt',
        'returns_qty',
        'net_sales_qty',
        'weekday',
        'month',
        'year',
        'is_weekend',
        'is_return_flag',
        'is_price_missing',
    ]
]

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
mart_daily_sales.to_csv(PROCESSED_DATA_DIR / 'mart_daily_sales.csv', index=False)
mart_daily_sales.head()

## Что перенести в отчет

- число строк в исходном файле: `[A]`;
- доля возвратов: `[B]`;
- доля строк без цены: `[C]`;
- число строк в дневной витрине: `[D]`.